In [1]:
import glob
from itertools import product

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from kymatio.scattering2d.frontend.torch_frontend import ScatteringTorch2D
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader

from models.classical.CNN import CNN
from models.hybrid.ScatteringCNN import ScatteringCNN
from models.hybrid.ScatteringClassifier import ScatteringClassifier
from scattering import coefficients
from scripts.fit_models import get_folds
from scripts.preprocess import preprocess
from scripts.training_utils import TorchDataset, train_classifier, evaluate_classifier
from wavelets import SolidHarmonicModel

# Dataset
### Classification

In [ ]:
M = 36
N = M
tng_files = glob.glob("../datasets/tng_as_sdss/*fits")

logmass = True

save_path = f"./galaxy-classification/changed-non-mergers-{M}X{N}.npz"

X, Y_ratio, Y_time, Y_is_merger, Y_stellar_mass = preprocess(tng_files, save_path, M, N)

print(X.shape)
print("Mergers", Y_is_merger.sum())
print("Non-Mergers", len(Y_is_merger) - Y_is_merger.sum())

Loading saved data...
(13788, 3, 36, 36)
Mergers 6807
Non-Mergers 6981


In [3]:
def augment_perspective(image):
    C, H, W = image.shape
    deformed = np.zeros_like(image)
    for c in range(C):
        src_points = np.array([
            [0, 0],
            [W - 1, 0],
            [0, H - 1],
            [W - 1, H - 1]
        ], dtype=np.float32)

        scale = 0.15
        dst_points = src_points + np.random.uniform(-scale * W, scale * W, size=(4, 2))

        M = cv2.getPerspectiveTransform(src_points, dst_points.astype(np.float32))
        deformed[c] = cv2.warpPerspective(image[c], M, (W, H), borderMode=cv2.INTER_LANCZOS4)

    return deformed


In [4]:
# SOURCE: Mathilda Avirett-Mackenzie, modified by Alex Brown
def get_random_d4_augs(img, n_aug=1, replace=True, seed=None):
    img = np.transpose(img, (1, 2, 0))  # (C, M, N) to (M, N, C)
    rng = np.random.default_rng(seed=seed)

    # whether to reflect and number of rotations
    aug_list = [(False, 0), (True, 0), (False, 1), (True, 1),
                (False, 2), (True, 2), (False, 3), (True, 3)]

    # forces replacement if more than 8 augmentations are requested
    if n_aug > 8:
        replace = True

    augs_used = rng.choice(aug_list, size=n_aug, replace=replace)

    output = []

    for (reflect, n_rot) in augs_used:
        # initialise image to add to output list
        add_img = img

        if reflect:
            add_img = np.flip(add_img, axis=0)

        for i in range(n_rot):
            add_img = np.rot90(add_img, axes=(0, 1))

        add_img = np.transpose(add_img, (2, 0, 1))  # (M, N, C) to (C, M, N)
        output.append(add_img)

    return output


# SOURCE: Mathilda Avirett-Mackenzie
def get_n_aug(log_mass):
    if log_mass < 10:
        return 1
    if log_mass < 11:
        return 2
    if log_mass < 11.5:
        return 3
    return 8


# SOURCE: Alex Brown
def boost(data, target, y_stellar):
    X_boosted, Y_merger_boosted, Y_stellar_boosted = [], [], []
    for i, img in enumerate(data):
        log_mass = np.log10(y_stellar[i]) + 10
        n_aug = get_n_aug(log_mass)
        aug_imgs = get_random_d4_augs(img, n_aug)
        for aug_img in aug_imgs:
            X_boosted.append(aug_img)
            Y_merger_boosted.append(target[i])
            Y_stellar_boosted.append(y_stellar[i])

    return np.array(X_boosted), np.array(Y_merger_boosted), np.array(Y_stellar_boosted)


def boost_perspective(data, target, y_stellar):
    X_boosted, Y_merger_boosted, Y_stellar_boosted = [], [], []
    for i, img in enumerate(data):
        log_mass = np.log10(y_stellar[i]) + 10
        n_aug = get_n_aug(log_mass)
        for y in range(n_aug):
            X_boosted.append(augment_perspective(img))
            Y_merger_boosted.append(target[i])
            Y_stellar_boosted.append(y_stellar[i])

    return np.array(X_boosted), np.array(Y_merger_boosted), np.array(Y_stellar_boosted)


# PyTorch models

## Classical CNN

In [5]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

boosted = True
random_state = 42

Y_stacked = np.stack([Y_is_merger, Y_stellar_mass], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, Y_stacked, test_size=0.2, stratify=Y_is_merger,
                                                    random_state=random_state)

X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5, stratify=y_test[:, 0],
                                                random_state=random_state)

y_stellar_train_original = y_train[:, 1].copy()
y_stellar_val_original = y_val[:, 1].copy()

y_stellar_train = y_train[:, 1]
y_train = y_train[:, 0]

y_stellar_val = y_val[:, 1]
y_val = y_val[:, 0]
y_test = y_test[:, 0]

if boosted:
    X_train, y_train, y_train_stellar = boost(X_train, y_train, y_stellar_train)
    X_val, y_val, y_val_stellar = boost(X_val, y_val, y_stellar_val)

batch_size = 32

train_dataset = TorchDataset(X_train, y_train, classification=True)
val_dataset = TorchDataset(X_val, y_val, augmentations=None, classification=True)
test_dataset = TorchDataset(X_test, y_test, augmentations=None, classification=True)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

bins = 16
alpha = 0.7

y_stellar_train_original = np.log10(y_stellar_train_original) + 10
y_stellar_val_original = np.log10(y_stellar_val_original) + 10
y_val_stellar = np.log10(y_val_stellar) + 10
y_train_stellar = np.log10(y_train_stellar) + 10

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Training Distribution
axes[0].hist(y_stellar_train_original, bins=bins, alpha=alpha, color='steelblue',
             edgecolor='black', label='Before Boosting')
axes[0].hist(y_train_stellar, bins=bins, alpha=alpha, color='crimson',
             edgecolor='black', label='After Boosting')
axes[0].set_title('Training Set: Stellar Mass Distribution', fontsize=14)
axes[0].set_xlabel(r'$\log(M_{\star} / M_{\odot})$', fontsize=13)
axes[0].set_ylabel(r'$N_{\mathrm{gal}}$', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation Distribution
axes[1].hist(y_stellar_val_original, bins=bins, alpha=alpha, color='seagreen',
             edgecolor='black', label='Before Boosting')
axes[1].hist(y_val_stellar, bins=bins, alpha=alpha, color='darkorange',
             edgecolor='black', label='After Boosting')
axes[1].set_title('Validation Set: Stellar Mass Distribution', fontsize=14)
axes[1].set_xlabel(r'$\log(M_{\star} / M_{\odot})$', fontsize=13)
axes[1].set_ylabel(r'$N_{\mathrm{gal}}$', fontsize=13)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

learning_rate = 0.001
patience = 7
min_lr = 1e-6
early_stopping = 25

model = CNN(input_shape=(3, M, M))
model.to(device)

optimiser = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiser, patience=patience, min_lr=min_lr, factor=0.5)

criterion = torch.nn.CrossEntropyLoss()

n_epochs = 500

trained_model = train_classifier(model, device, train_dataloader, val_dataloader, optimiser, scheduler, criterion,
                                 n_epochs, early_stopping=early_stopping)

print(f"\n-------------\nTEST: {random_state}\n-------------")
test_acc = evaluate_classifier(trained_model[0], device, test_dataloader)

history = trained_model[1]
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.legend()
plt.show()

plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')
plt.legend()
plt.show()

Instantiated model with 1153634 params.
[1/500]. Train Loss: 0.6634. Train accuracy: 0.6072. Val Loss: 0.5925. Val accuracy: 0.6827
Val loss decreased from inf to 0.5925. Saving new model for val acc: 0.6827171444892883.
[2/500]. Train Loss: 0.6318. Train accuracy: 0.6096. Val Loss: 0.6083. Val accuracy: 0.6707
[3/500]. Train Loss: 0.6152. Train accuracy: 0.6488. Val Loss: 0.6326. Val accuracy: 0.6526
[4/500]. Train Loss: 0.6049. Train accuracy: 0.6538. Val Loss: 0.5948. Val accuracy: 0.6767
[5/500]. Train Loss: 0.6034. Train accuracy: 0.6538. Val Loss: 0.5815. Val accuracy: 0.6930
Val loss decreased from 0.5925 to 0.5815. Saving new model for val acc: 0.6930353045463562.
[6/500]. Train Loss: 0.5990. Train accuracy: 0.6569. Val Loss: 0.5862. Val accuracy: 0.6819
[7/500]. Train Loss: 0.6007. Train accuracy: 0.6574. Val Loss: 0.5779. Val accuracy: 0.6836
Val loss decreased from 0.5815 to 0.5779. Saving new model for val acc: 0.6835770010948181.
[8/500]. Train Loss: 0.5935. Train accuracy

AttributeError: 'tuple' object has no attribute 'modules'

## Hybrid

In [14]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
boosted = True

Y_stacked = np.stack([Y_is_merger, Y_stellar_mass], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, Y_stacked, test_size=0.2, stratify=Y_is_merger, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5, stratify=y_test[:, 0], random_state=42)

y_stellar_train_original = y_train[:, 1].copy()
y_stellar_val_original = y_val[:, 1].copy()

y_stellar_train = y_train[:, 1]
y_train = y_train[:, 0]

y_stellar_val = y_val[:, 1]
y_val = y_val[:, 0]
y_test = y_test[:, 0]

if boosted:
    X_train, y_train, y_train_stellar = boost(X_train, y_train, y_stellar_train)
    X_val, y_val, y_val_stellar = boost(X_val, y_val, y_stellar_val)

batch_size = 64

train_dataset = TorchDataset(X_train, y_train, classification=True)
val_dataset = TorchDataset(X_val, y_val, augmentations=None, classification=True)
test_dataset = TorchDataset(X_test, y_test, augmentations=None, classification=True)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

bins = 16
alpha = 0.7

y_stellar_train_original = np.log10(y_stellar_train_original) + 10
y_stellar_val_original = np.log10(y_stellar_val_original) + 10
y_val_stellar = np.log10(y_val_stellar) + 10
y_train_stellar = np.log10(y_train_stellar) + 10

learning_rate = 0.001
patience = 15
min_lr = 1e-6
early_stopping = 50

scattering = ScatteringTorch2D(J=2, shape=(36, 36), L=8)
model = ScatteringCNN(
    input_shape=(36, 36),
    scattering=scattering,
    feature_dropout=0.05,
    classifier_dropout=0.05
)
model.to(device)

optimiser = torch.optim.AdamW(model.parameters(), lr=learning_rate)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiser, patience=patience, min_lr=min_lr, factor=0.5)
criterion = torch.nn.CrossEntropyLoss()

n_epochs = 500

trained_model = train_classifier(model, device, train_dataloader, val_dataloader, optimiser, scheduler, criterion,
                                 n_epochs, early_stopping=early_stopping)

print("\n-------------\nTEST\n-------------")
test_accuracy = evaluate_classifier(trained_model[0], device, test_dataloader)
history = trained_model[1]
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.legend()
plt.show()

plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')
plt.legend()
plt.show()

## Solid Harm

In [15]:
learning_rate = 0.01
patience = 5
min_lr = 1e-6
early_stopping = 25
dropout = 0.5

istc_units = 1024

scattering_config = {
    "L": 4,
    "J": 3,
    "sigma": 0.5,
    "lp_norms": [0.5, 1, 2, 3, 4],
    "bispectrum": False,
    "bicoherence": True,
    "reduce": True
}
higher = scattering_config.copy()

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")

boosted = True

Y_stacked = np.stack([Y_is_merger, Y_stellar_mass], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, Y_stacked, test_size=0.2, stratify=Y_is_merger, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5, stratify=y_test[:, 0], random_state=42)

y_stellar_train_original = y_train[:, 1].copy()
y_stellar_val_original = y_val[:, 1].copy()

y_stellar_train = y_train[:, 1]
y_train = y_train[:, 0]

y_stellar_val = y_val[:, 1]
y_val = y_val[:, 0]
y_test = y_test[:, 0]

if boosted:
    X_train, y_train, y_train_stellar = boost(X_train, y_train, y_stellar_train)
    X_val, y_val, y_val_stellar = boost(X_val, y_val, y_stellar_val)

batch_size = 32

train_dataset = TorchDataset(X_train, y_train, scattering=False, classification=True)
val_dataset = TorchDataset(X_val, y_val, augmentations=None, classification=True)
test_dataset = TorchDataset(X_test, y_test, augmentations=None, classification=True)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

bins = 16
alpha = 0.7

y_stellar_train_original = np.log10(y_stellar_train_original) + 10
y_stellar_val_original = np.log10(y_stellar_val_original) + 10
y_val_stellar = np.log10(y_val_stellar) + 10
y_train_stellar = np.log10(y_train_stellar) + 10

scattering = SolidHarmonicModel(36, 36, scattering_config, higher_order_config=higher)
model = ScatteringClassifier(input_shape=(36, 36), scattering=scattering, reduced_units=istc_units, dropout=dropout,
                             output_type="c")
model.to(device)

optimiser = torch.optim.AdamW(model.parameters(), lr=learning_rate)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiser, patience=patience, min_lr=min_lr)
criterion = torch.nn.CrossEntropyLoss()

n_epochs = 500

trained_model = train_classifier(model, device, train_dataloader, val_dataloader, optimiser, scheduler, criterion,
                                 n_epochs, early_stopping=early_stopping)

print("\n-------------\nTEST\n-------------")
test_accuracy = evaluate_classifier(trained_model[0], device, test_dataloader)
history = trained_model[1]
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.legend()
plt.show()

plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')
plt.legend()
plt.show()

# Sklearn

In [18]:
param_grid = {
}

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ("classifier", RidgeClassifier(solver="svd"))
])
boosted = True

Y_stacked = np.stack([Y_is_merger, Y_stellar_mass], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, Y_stacked, test_size=0.2, stratify=Y_is_merger, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5, stratify=y_test[:, 0], random_state=42)

y_stellar_train_original = y_train[:, 1].copy()
y_stellar_val_original = y_val[:, 1].copy()

y_stellar_train = y_train[:, 1]
y_train = y_train[:, 0]

y_stellar_val = y_val[:, 1]
y_val = y_val[:, 0]
y_test = y_test[:, 0]

if boosted:
    X_train, y_train, y_train_stellar = boost(X_train, y_train, y_stellar_train)
    X_val, y_val, y_val_stellar = boost(X_val, y_val, y_stellar_val)

X_train = np.concatenate([X_train, X_val])
y_train = np.concatenate([y_train, y_val])
L = [2, 4, 6, 8, 10]
J = [2, 3]
sigma = [0.5]
lp_norms = [[0.25, 0.5, 0.75, 1, 2, 3, 4]]

for l, j, sig, lp in product(L, J, sigma, lp_norms):
    hos = "scattering"
    print(l, j, sig, lp)
    scattering_config = {
        "L": l,
        "J": j,
        "sigma": sig,
        "lp_norms": lp,
        "bispectrum": False,
        "bicoherence": True,
        "reduce": True
    }
    higher_config = {
        "L": l,
        "J": j,
        "sigma": sig,
        "lp_norms": lp,
        "bicoherence": True,
    }
    #
    train_coef, train_y = coefficients.calculate_coefficients(X_train, y_train, M, N, scattering_config, higher_config,
                                                              f"galaxy-classification-train-val-{M}x{N}-{hos}-{boosted}",
                                                              hos)

    train_coef = train_coef[:-3]
    train_y = train_y[:-3]

    print(train_coef.shape, train_y.shape)
    test_coef, test_y = coefficients.calculate_coefficients(X_test, y_test, M, N, scattering_config, higher_config,
                                                            f"galaxy-classification-test-{M}x{N}-{hos}-{boosted}", hos)
    # X_test = coefficients.calculate_coefficients_morlet(X_test, M, N, l, j,
    #                                                      f"morlet-galaxy-classification-test-{M}x{N}-{hos}-{boosted}", hos)
    test_coef = test_coef.reshape(test_coef.shape[0], -1)

    cross_val_folds = get_folds(train_coef)

    model = GridSearchCV(pipeline, param_grid, cv=cross_val_folds).fit(train_coef, train_y)

    y_pred = model.predict(test_coef)
    acc = accuracy_score(test_y, y_pred)
    prec = precision_score(test_y, y_pred, average="weighted")
    recall = recall_score(test_y, y_pred, average="weighted")
    f1 = f1_score(test_y, y_pred, average="weighted")

    best_params = model.best_params_
    print(f"Best params: {best_params}, Accuracy: {acc}, Precision: {prec}, Recall: {recall}, F1: {f1}")